In [1]:
import pandas as pd
import re, unicodedata

In [4]:
df = pd.read_csv("hospital.csv")

In [5]:
df.head(10)

,Feedback,Sentiment Label,Ratings
0,Good and clean hospital. There is great team o...,1,5
1,Had a really bad experience during discharge. ...,1,5
2,I have visited to take my second dose and Proc...,1,4
3,That person was slightly clueless and offered...,1,3
4,There is great team of doctors and good OT fac...,0,1
5,My primary concern arose from the insistence o...,0,2
6,Good and clean hospital. The medical faciliti...,1,5
7,Recently underwent a surgery for my left shoul...,1,3
8,"Over all experience was good, starting from re...",1,5
9,"However,the services of front office (where we...",1,5


In [6]:
# standardise column names
df = df.rename(columns=lambda c: c.strip().lower().replace(" ", "_"))

In [7]:
# to map other columns according to our table schema

col_map = {
    "feedback": "feedback",
    "review": "feedback",
    "text": "feedback",
    "comment": "feedback",
    "ratings": "rating",
    "rating": "rating",
    "sentiment_label": "sentiment_label",
    "sentiment": "sentiment_label",
    "date": "date",
    "clinic_id": "clinic_id",
    "hospital_id": "clinic_id",
}
df = df.rename(columns={c: col_map.get(c, c) for c in df.columns})

In [8]:
# checking required columns

required = ["feedback", "rating"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found: {list(df.columns)}")

In [9]:
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = unicodedata.normalize("NFKC", x)      # normalize unicode
    x = x.replace("\xa0", " ").replace("Â", "")  # removing weird charachters
    x = re.sub(r"\s+", " ", x).strip()        # collapse spaces
    return x.lower()

df["feedback_raw"] = df["feedback"].astype(str)
df["feedback"] = df["feedback"].apply(clean_text)

In [10]:
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df = df.dropna(subset=["rating"])
df["rating"] = df["rating"].astype(int)
df = df[(df["rating"] >= 1) & (df["rating"] <= 5)]

In [11]:
# dropping empty feedbacks and ratings
df = df[df["feedback"].str.len() > 0]
dedup_keys = ["feedback", "rating"]
if "clinic_id" in df.columns: dedup_keys.append("clinic_id")
df = df.drop_duplicates(subset=dedup_keys).reset_index(drop=True)

In [13]:
df.head(10)

,feedback,sentiment_label,rating,feedback_raw
0,good and clean hospital. there is great team o...,1,5,Good and clean hospital. There is great team o...
1,had a really bad experience during discharge. ...,1,5,Had a really bad experience during discharge. ...
2,i have visited to take my second dose and proc...,1,4,I have visited to take my second dose and Proc...
3,that person was slightly clueless and offered ...,1,3,That person was slightly clueless and offered...
4,there is great team of doctors and good ot fac...,0,1,There is great team of doctors and good OT fac...
5,my primary concern arose from the insistence o...,0,2,My primary concern arose from the insistence o...
6,good and clean hospital. the medical facilitie...,1,5,Good and clean hospital. The medical faciliti...
7,recently underwent a surgery for my left shoul...,1,3,Recently underwent a surgery for my left shoul...
8,"over all experience was good, starting from re...",1,5,"Over all experience was good, starting from re..."
9,"however,the services of front office (where we...",1,5,"However,the services of front office (where we..."


In [14]:
df["rating"].value_counts().sort_index()

rating
1    120
2    145
3    120
4    250
5    341
Name: count, dtype: int64

In [15]:
df["feedback"].str.contains("Â").sum()

0

In [16]:
!pip -q install nltk
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")  


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Meow\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Meow\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [17]:
from nltk.tokenize import sent_tokenize

In [18]:
# takes reviews and splits it into sentences, removes spaces and very tiny outputs

def split_into_sentences(text):
    sents = sent_tokenize(text)
    sents = [s.strip() for s in sents if len(s.strip()) >= 4]
    return sents

In [19]:
# creates a unique review id for each sentence, so we can track which sentence came from which review
df = df.reset_index(drop=True).copy()
df["review_id"] = df.index  


# creates one row per review sentence
rows = []
for _, r in df.iterrows():
    for sent in split_into_sentences(r["feedback_raw"]):  # use raw for nicer evidence
        rows.append({
            "review_id": r["review_id"],
            "chunk_text": sent.strip(),
            "rating": r["rating"],
            "sentiment_label": r.get("sentiment_label", None)
        })

chunks = pd.DataFrame(rows)

In [21]:
chunks.sample(15, random_state=42)

,review_id,chunk_text,rating,sentiment_label
614,373,They provide good and quality care Nurses are ...,5,1
174,98,Big thanks to Dr Narayan Hulse.,5,1
869,609,The worst experience that I ever had at the ho...,1,0
941,681,Very good and fast service.,4,1
980,720,We were at manipal to get my father treated fo...,5,1
759,511,No support for patients after consultation hou...,1,0
874,614,adiology department is very clean and give goo...,4,1
78,42,Thank you So much !!,5,1
254,138,In emergency there was only 1 nurse who was no...,2,0
1136,876,NU Hospitals believes in patient centricity an...,4,1


In [22]:
print("Reviews:", len(df))
print("Chunks:", len(chunks))

Reviews: 976
Chunks: 1238


Reviews talk about the same issue in many different ways:

“Waited 2 hours”

“Appointment delayed”

“Long queue at reception”

Keyword matching might treat these as different. Embeddings convert each sentence into a vector that captures meaning, so those sentences end up near each other in vector space.
Then clustering groups “nearby meaning” sentences into themes automatically.

That gives you:

theme groups (topic clusters)

how often each theme appears (frequency)

evidence sentences per theme (explainability)

In [24]:
!pip -q install sentence-transformers scikit-learn


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans

In [26]:
# turning sentence into vectors

model = SentenceTransformer("all-MiniLM-L6-v2")  # fast + good for hackathon
X = model.encode(chunks["chunk_text"].tolist(), normalize_embeddings=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Meow\anaconda3\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Meow\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [30]:
# clustering similar sentences

K = 12  # start with 10–20 themes for hackathon
kmeans = MiniBatchKMeans(n_clusters=K, random_state=42, batch_size=2048)
labels = kmeans.fit_predict(X)

chunks["cluster_id"] = labels

In [51]:
# theme frequency distribution

theme_freq = (
    chunks[chunks["cluster_id"] != -1]["cluster_id"]
    .value_counts(normalize=True)
    .mul(100).round(1)
    .reset_index()
    .rename(columns={"index": "cluster_id", "cluster_id": "frequency_%"})
)
theme_freq.head(10)

,frequency_%,proportion
0,3,15.4
1,1,13.3
2,7,11.9
3,5,9.7
4,8,8.2
5,4,7.4
6,6,7.4
7,0,6.3
8,9,5.8
9,11,5.7


In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer

def top_keywords(texts, k=6):
    v = TfidfVectorizer(stop_words="english", ngram_range=(1,2), max_features=8000)
    M = v.fit_transform(texts)
    scores = np.asarray(M.mean(axis=0)).ravel()
    idx = scores.argsort()[::-1][:k]
    return [v.get_feature_names_out()[i] for i in idx]

cluster_names = {}
for cid in sorted(chunks["cluster_id"].unique()):
    texts = chunks.loc[chunks["cluster_id"] == cid, "chunk_text"].tolist()
    cluster_names[cid] = " / ".join(top_keywords(texts, k=4))

chunks["theme_label_auto"] = chunks["cluster_id"].map(cluster_names)
chunks[["chunk_text", "cluster_id", "theme_label_auto"]].head(10)

,chunk_text,cluster_id,theme_label_auto
0,Good and clean hospital.,3,hospital / good / service / doctors
1,There is great team of doctors and good OT fac...,3,hospital / good / service / doctors
2,The medical facilities are all great with good...,3,hospital / good / service / doctors
3,The housekeeping staff is also good but they c...,4,staff / good / maintained / service
4,Had a really bad experience during discharge.,6,wait / doctor / time / test
5,They need to be sensitive and more transparent...,1,hospital / worst / patient / doctors
6,I have visited to take my second dose and Proc...,6,wait / doctor / time / test
7,Hospitality from all staffs are really appreci...,4,staff / good / maintained / service
8,Shanti tooks good care and provide all details...,8,treatment / good / hospital / thanks
9,Thank you.,0,thank / good / thanks / caring


In [38]:
evidence = (
    chunks[chunks["cluster_id"] != -1]
    .groupby("cluster_id")["chunk_text"]
    .apply(lambda s: s.head(10).tolist())
    .to_dict()
)
evidence

{0: ['Thank you So much !!',
  'Thanks to energetic and charming Nurse Ms Jacqueline for special care taken while treatment !',
  "Each one's role and efforts are much appreciated..",
  'She became totally bedridden.',
  'He gave us 100% assurance and operated her.',
  'She walks now independently of course with Walker.',
  'She is ever grateful to him.',
  'Friend finished the process by 11:30.',
  'Wife finished at 4 pm.',
  'Special thanks to Kavita for all the guidance and support she had provided us so far.'],
 1: ['They need to be sensitive and more transparent towards the patient and his/her family.',
  'But once i got to the hospital, I found a better and more economical one and decided to go ahead',
  'Recently underwent a surgery for my left shoulder, the doctors are extremely good, but the management needs to be more patient oriented than money minded.',
  'Hopeless hospital.',
  'They have no value for patients or their attendants time.',
  "They don't care if the doctor is

In [36]:
chunks = chunks[~chunks["chunk_text"].str.lower().isin(["thank you.", "thank you", "thanks"])]

In [37]:
custom_stop = ["hospital","doctor","doctors","good","service","staff","thanks","thank","patient"]
v = TfidfVectorizer(stop_words="english", ngram_range=(1,2), max_features=8000)

In [39]:
chunks["cluster_id"].value_counts()

cluster_id
3     191
1     165
7     147
5     120
8     102
4      92
6      92
0      78
9      72
11     70
2      55
10     53
Name: count, dtype: int64

In [57]:
df.groupby("sentiment_label")["rating"].agg(["count","mean"])

,count,mean
sentiment_label,,
0,265,1.54717
1,711,4.31083


In [52]:
!pip -q install transformers torch


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [56]:
from transformers import pipeline
clf = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

def severity(text):
    out = clf(text[:512])[0]
    # label is POSITIVE/NEGATIVE
    return out["score"] if out["label"] == "NEGATIVE" else 1 - out["score"]

chunks["severity_score"] = chunks["chunk_text"].apply(severity)
chunks[["chunk_text","severity_score"]].head(25)

Device set to use cpu


,chunk_text,severity_score
0,Good and clean hospital.,0.000264
1,There is great team of doctors and good OT fac...,0.000326
2,The medical facilities are all great with good...,0.000197
3,The housekeeping staff is also good but they c...,0.000574
4,Had a really bad experience during discharge.,0.999627
5,They need to be sensitive and more transparent...,0.981581
6,I have visited to take my second dose and Proc...,0.000561
7,Hospitality from all staffs are really appreci...,0.000225
8,Shanti tooks good care and provide all details...,0.003016
10,That person was slightly clueless and offered ...,0.999050


In [58]:
import pandas as pd
import numpy as np

K = chunks["cluster_id"].nunique()

# total chunks per review
tot = chunks.groupby("review_id").size().rename("n_chunks")

# freq per theme per review
freq = (chunks
        .groupby(["review_id","cluster_id"])
        .size()
        .div(tot)
        .rename("freq")
        .reset_index())

# avg severity per theme per review
sev = (chunks
       .groupby(["review_id","cluster_id"])["severity_score"]
       .mean()
       .rename("sev")
       .reset_index())

feat_long = freq.merge(sev, on=["review_id","cluster_id"], how="left")
feat_long["intensity"] = feat_long["freq"] * feat_long["sev"]

# wide matrix: one row per review, one col per theme
X = feat_long.pivot_table(index="review_id", columns="cluster_id", values="intensity", fill_value=0)
X.columns = [f"theme_{c}_intensity" for c in X.columns]

y = df.set_index("review_id").loc[X.index, "rating"]

In [59]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np

reg = make_pipeline(StandardScaler(with_mean=False), RidgeCV(alphas=np.logspace(-3,3,20)))

cv = KFold(n_splits=5, shuffle=True, random_state=42)
mae = -cross_val_score(reg, X, y, cv=cv, scoring="neg_mean_absolute_error").mean()
print("CV MAE:", round(mae, 3))

reg.fit(X, y)

# pull coefficients
ridge = reg.named_steps["ridgecv"]
coef = pd.Series(ridge.coef_, index=X.columns).sort_values()
coef.head(10), coef.tail(10)

CV MAE: 0.708


(theme_1_intensity    -0.808007
 theme_6_intensity    -0.582912
 theme_9_intensity    -0.470244
 theme_11_intensity   -0.455621
 theme_4_intensity    -0.302687
 theme_7_intensity    -0.222314
 theme_3_intensity    -0.218679
 theme_2_intensity    -0.182377
 theme_5_intensity    -0.155183
 theme_8_intensity    -0.153126
 dtype: float64,
 theme_9_intensity    -0.470244
 theme_11_intensity   -0.455621
 theme_4_intensity    -0.302687
 theme_7_intensity    -0.222314
 theme_3_intensity    -0.218679
 theme_2_intensity    -0.182377
 theme_5_intensity    -0.155183
 theme_8_intensity    -0.153126
 theme_0_intensity    -0.119421
 theme_10_intensity   -0.088021
 dtype: float64)

In [60]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

mask = y.isin([1,5])
X_ext = X[mask]
y_ext = (y[mask] == 1).astype(int)  # 1=bad(1-star), 0=good(5-star)

clf = make_pipeline(
    StandardScaler(with_mean=False),
    LogisticRegression(max_iter=2000, class_weight="balanced")
)

cv2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc = cross_val_score(clf, X_ext, y_ext, cv=cv2, scoring="roc_auc").mean()
print("CV AUC:", round(auc, 3))

clf.fit(X_ext, y_ext)
log = clf.named_steps["logisticregression"]
imp = pd.Series(log.coef_[0], index=X.columns).sort_values(ascending=False)
imp.head(10)

CV AUC: 0.921


theme_1_intensity     1.619548
theme_6_intensity     1.253951
theme_9_intensity     1.085754
theme_11_intensity    0.733486
theme_4_intensity     0.585225
theme_7_intensity     0.518103
theme_2_intensity     0.419449
theme_0_intensity     0.334261
theme_3_intensity     0.319600
theme_5_intensity     0.313327
dtype: float64

In [61]:
impact_table = pd.DataFrame({
    "feature": coef.index,
    "rating_impact_coef": coef.values
}).sort_values("rating_impact_coef")

impact_table.head(15)

,feature,rating_impact_coef
0,theme_1_intensity,-0.808007
1,theme_6_intensity,-0.582912
2,theme_9_intensity,-0.470244
3,theme_11_intensity,-0.455621
4,theme_4_intensity,-0.302687
5,theme_7_intensity,-0.222314
6,theme_3_intensity,-0.218679
7,theme_2_intensity,-0.182377
8,theme_5_intensity,-0.155183
9,theme_8_intensity,-0.153126


In [63]:
import pandas as pd
import numpy as np

cluster_summary = (
    chunks.groupby("cluster_id")
    .agg(
        n_chunks=("chunk_text", "count"),
        avg_severity=("severity_score", "mean"),
        avg_rating=("rating", "mean")
    )
    .reset_index()
)

cluster_summary["freq_%"] = (cluster_summary["n_chunks"] / len(chunks) * 100).round(1)
cluster_summary.sort_values("n_chunks", ascending=False).head(10)

,cluster_id,n_chunks,avg_severity,avg_rating,freq_%
3,3,191,0.108611,4.235602,15.4
1,1,165,0.889438,2.157576,13.3
7,7,147,0.134447,4.367347,11.9
5,5,120,0.065517,4.358333,9.7
8,8,102,0.154443,4.264706,8.2
4,4,92,0.238054,4.021739,7.4
6,6,92,0.836972,2.336957,7.4
0,0,78,0.139068,4.217949,6.3
9,9,72,0.745876,2.347222,5.8
11,11,70,0.862006,2.585714,5.7


In [72]:
def show_cluster_examples(cid, n=7):
    ex = chunks[chunks["cluster_id"]==cid].copy()
    ex = ex.sort_values("severity_score", ascending=False) 
    return ex[["chunk_text","rating","sentiment_label","severity_score"]].head(n)

show_cluster_examples(4, n=10)

,chunk_text,rating,sentiment_label,severity_score
881,The staff seem to have become less responsive ...,1,0,0.999797
971,The staff seem to have become less responsive ...,3,1,0.999797
535,The staff is not at all helpful and do not giv...,1,0,0.999790
951,Facility staff and cleanliness is missing.,1,0,0.999787
1082,Worst staff please don't go that lab my person...,2,0,0.999781
1182,Worst staff please don't go that lab my person...,1,0,0.999781
1179,All staffs are not friendly so that we can fe...,2,0,0.999651
500,the food is not tasty and is not made well Thi...,2,0,0.999541
1076,No helping nature no support All lab staff are...,1,0,0.999177
502,Only concern is with the admission process and...,3,1,0.998993
